# 7. Weighted Importance Sampling
### Corrects Off-Policy Estimates Using Weighted Returns

**Importance Sampling** is a statistical technique for estimating properties of a target distribution $\pi$ given samples generated from a behavior distribution $b$.

#### Mathematical Comparison:
1. **Importance Sampling Ratio**:
   $$\rho_{t:T-1} = \prod_{k=t}^{T-1} \frac{\pi(A_k \mid S_k)}{b(A_k \mid S_k)}$$
2. **Ordinary Importance Sampling**:
   $$V(s) = \frac{\sum_{i \in \mathcal{T}(s)} \rho_{t_i:T_i-1} G_{t_i}}{|\mathcal{T}(s)|}$$
   - **Unbiased**, but has **potentially infinite variance**.
3. **Weighted Importance Sampling**:
   $$V(s) = \frac{\sum_{i \in \mathcal{T}(s)} \rho_{t_i:T_i-1} G_{t_i}}{\sum_{i \in \mathcal{T}(s)} \rho_{t_i:T_i-1}}$$
   - **Biased** (bias approaches 0 as $N \to \infty$), but has **guaranteed finite, bounded variance**.


In [ ]:
# Step 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# Step 2: Canonical Infinite Variance Experiment (Sutton & Barto Example 5.5)
# State: Single non-terminal state with 2 actions:
# Action 0 (back to start, prob 0.9 under b), Action 1 (terminate with reward +1, prob 0.1 under b)
# Target policy π always takes Action 0.

def run_is_simulation(num_runs=20, num_episodes=5000):
    ordinary_errors = np.zeros((num_runs, num_episodes))
    weighted_errors = np.zeros((num_runs, num_episodes))
    true_value = 1.0 # True discounted value under π with γ=1
    
    for r in range(num_runs):
        ord_num = 0.0
        weight_num = 0.0
        weight_den = 0.0
        
        for ep in range(num_episodes):
            # Generate episode under behavior policy b (prob 0.5 for each action)
            actions = []
            while True:
                a = np.random.choice([0, 1])
                actions.append(a)
                if a == 1:
                    break
                    
            # Compute importance ratio: π always takes 0, b takes 0 with prob 0.5
            # If length is L, actions are [0, 0, ..., 1]
            # Since π(1) = 0, if last action is 1, π(1)/b(1) = 0/0.5 = 0!
            # The only trajectory with non-zero weight under π is ending at step 1
            if actions == [1]:
                W = (1.0 / 0.5) ** len(actions) # Ratio for the trajectory
                G = 1.0
            else:
                W = 0.0
                G = 1.0
                
            ord_num += W * G
            weight_num += W * G
            weight_den += W
            
            # Estimates
            ord_est = ord_num / (ep + 1)
            weight_est = (weight_num / weight_den) if weight_den > 0 else 0.0
            
            ordinary_errors[r, ep] = (ord_est - true_value) ** 2
            weighted_errors[r, ep] = (weight_est - true_value) ** 2
            
    return np.mean(ordinary_errors, axis=0), np.mean(weighted_errors, axis=0)

ord_mse, weighted_mse = run_is_simulation(num_runs=15, num_episodes=2000)


In [ ]:
# Step 3: Plot Mean Squared Error: Ordinary vs Weighted Importance Sampling
plt.figure(figsize=(9, 5))
plt.plot(ord_mse, label='Ordinary Importance Sampling (High/Infinite Variance)', color='red', alpha=0.8)
plt.plot(weighted_mse, label='Weighted Importance Sampling (Bounded Variance)', color='blue', lw=2)
plt.yscale('log')
plt.xlabel("Episodes (log scale)")
plt.ylabel("Mean Squared Error (MSE)")
plt.title("Importance Sampling Variance: Ordinary vs Weighted")
plt.legend()
plt.grid(True, which='both', linestyle='--', alpha=0.7)
plt.show()


### Summary
- Ordinary Importance Sampling is strictly unbiased, but explosive variance often renders it unusable in practice.
- Weighted Importance Sampling dramatically reduces variance and is standard in off-policy Monte Carlo methods.
